# Tono · Fase 2 — cáncer de piel con auditoría por tono de piel

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental.

**PAD-UFES-20**: ~2.300 fotos tomadas con **teléfono móvil** (no dermatoscopio),
de ~1.370 pacientes de Brasil, con tipo de **Fitzpatrick**, identificador de
paciente y marca de confirmación por biopsia.

Tarea binaria: **maligno** (BCC, SCC, MEL) frente a **benigno** (ACK, NEV, SEK).
Es la pregunta que importa —¿esto hay que biopsiarlo?— y no una taxonomía de
seis clases.

La métrica principal **no es el AUROC global**: es la **brecha de falsos
negativos entre tonos de piel**. Un falso negativo aquí es un cáncer que se deja
pasar.

Código: [tono](https://github.com/GGGuardin/tono) · [pipeline](https://github.com/GGGuardin/chest-xray-pneumonia)

In [ ]:
import subprocess, sys, os, time, glob, json
T0 = time.time()

# Dos repos: `tono` aporta el lector de PAD-UFES; el de tórax aporta el pipeline
# de entrenamiento, evaluación y equidad, que consume un manifiesto y por tanto
# es independiente del dataset. No se reescribe lo que ya está probado.
for nombre, url in [('tono', 'https://github.com/GGGuardin/tono.git'),
                    ('cxr', 'https://github.com/GGGuardin/chest-xray-pneumonia.git')]:
    subprocess.run(['rm', '-rf', f'/tmp/{nombre}'], check=False)
    subprocess.run(['git', 'clone', '--depth', '1', '-q', url, f'/tmp/{nombre}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'], check=True)

import torch
cap = torch.cuda.get_device_capability(0)
assert f'sm_{cap[0]}{cap[1]}' in torch.cuda.get_arch_list(), 'GPU no soportada, relanza con T4'
torch.zeros(8, device='cuda').sum().item()
print('GPU:', torch.cuda.get_device_name(0), '| CUDA OK')
print('entradas:', os.listdir('/kaggle/input'))

## 1. Manifiesto y split POR PACIENTE

Una misma persona aporta varias lesiones, así que dividir por imagen metería
fotos de la misma piel y la misma cámara a ambos lados del split.

In [ ]:
OUT = '/kaggle/working'

# La ruta se descubre buscando un fichero ancla, no se codifica a mano
anclas = [p for p in glob.glob('/kaggle/input/**/*.csv', recursive=True)]
raices = sorted({os.path.dirname(p) for p in anclas})
print('candidatos:', raices[:6])
PAD = raices[0] if raices else '/kaggle/input'
# Si el CSV está en un subdirectorio, se sube al nivel que contiene las imágenes
while PAD != '/kaggle/input' and not glob.glob(os.path.join(PAD, '**', '*.png'), recursive=True):
    PAD = os.path.dirname(PAD)
print('PAD-UFES en:', PAD)

os.chdir('/tmp/tono')
!python -m datos.pad_ufes --root {PAD} --out {OUT}/manifiesto_pad.csv

In [ ]:
import pandas as pd

df = pd.read_csv(f'{OUT}/manifiesto_pad.csv')
print('Lesiones por paciente: media %.2f, maximo %d'
      % (len(df) / df.patient_id.nunique(), df.patient_id.value_counts().max()))
assert (df.groupby('patient_id')['split'].nunique() == 1).all(), 'FUGA DE DATOS'
print('OK: ningun paciente en mas de un split\n')

print('Prevalencia de malignidad por tono de Fitzpatrick:')
resumen = df.groupby('fitzpatrick').agg(
    n=('label', 'size'), malignos=('label', 'sum'), prevalencia=('label', 'mean'))
print(resumen.round(3).to_string())
print('\nOjo a los tamanos: los tonos con pocos casos daran intervalos anchos,')
print('y eso hay que decirlo en lugar de reportar la brecha como si fuera precisa.')

## 2. Entrenamiento

Se reutiliza el `train.py` del proyecto de tórax sin tocarlo: consume el
manifiesto. La única diferencia de dominio va en el config — aquí **sí** se
permite el volteo horizontal, porque una lesión cutánea no tiene lateralidad.

In [ ]:
os.chdir('/tmp/cxr')
!python -m src.train --config /tmp/tono/configs/pad_ufes.yaml \
    --manifest {OUT}/manifiesto_pad.csv \
    --out-dir {OUT}/runs/pad_densenet121

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs(f'{OUT}/reports', exist_ok=True)
h = pd.read_csv(f'{OUT}/runs/pad_densenet121/history.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label='train'); ax[0].plot(h.epoch, h.val_loss, label='val')
ax[0].set_title('loss'); ax[0].set_xlabel('epoca'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h.epoch, h.train_auroc, label='train'); ax[1].plot(h.epoch, h.val_auroc, label='val')
ax[1].set_title('AUROC'); ax[1].set_xlabel('epoca'); ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(f'{OUT}/reports/curvas_entrenamiento.png', dpi=150)
print(h.to_string(index=False))

## 3. Evaluación en test

In [ ]:
!python -m src.evaluate --checkpoint {OUT}/runs/pad_densenet121/best.pth \
    --manifest {OUT}/manifiesto_pad.csv --split test \
    --out-dir {OUT}/reports/test --n-boot 1000

## 4. Solo casos con biopsia

El subconjunto confirmado histológicamente es la única evaluación contra verdad
dura; el resto es impresión clínica. Se filtra sobre las predicciones ya
calculadas, sin repetir inferencia.

In [ ]:
sys.path.insert(0, '/tmp/cxr')
from src.metrics import binary_metrics, bootstrap_ci

pred = pd.read_csv(f'{OUT}/reports/test/predictions.csv')
umbral = float(json.load(open(f'{OUT}/reports/test/metrics.json'))['threshold'])

comparativa = {}
for nombre, sel in [('todos', pred.index),
                    ('biopsiados', pred.index[pred['biopsiado'].astype(str).str.upper()
                                              .isin(['TRUE', '1', 'YES'])])]:
    sub = pred.loc[sel]
    if sub['label'].nunique() < 2:
        print(f'{nombre}: sin ambas clases, se omite'); continue
    m = binary_metrics(sub['label'].values, sub['prob'].values, umbral)
    lo, hi = bootstrap_ci(sub['label'].values, sub['prob'].values, 'auroc',
                          n_boot=1000, threshold=umbral)
    comparativa[nombre] = {k: round(m[k], 4) for k in
                           ('n', 'prevalencia', 'auroc', 'auprc', 'sensibilidad',
                            'especificidad', 'fnr')}
    comparativa[nombre]['auroc_ci95'] = [round(lo, 4), round(hi, 4)]
    print(f"{nombre:12s} n={m['n']:4d}  AUROC {m['auroc']:.4f} [{lo:.4f}, {hi:.4f}]  "
          f"sens {m['sensibilidad']:.4f}  FNR {m['fnr']:.4f}")

json.dump(comparativa, open(f'{OUT}/reports/biopsia.json', 'w'), indent=2, ensure_ascii=False)

## 5. La razón de ser del proyecto: equidad por tono de piel

Tasa de falsos negativos por tipo de Fitzpatrick, por sexo, por grupo de edad y
por la intersección tono × sexo. **Un falso negativo es un cáncer que se deja
pasar**, así que es la métrica que decide si esta herramienta ayuda o perjudica.

In [ ]:
!python -m src.fairness --predictions {OUT}/reports/test/predictions.csv \
    --out-dir {OUT}/reports/equidad \
    --attributes fitzpatrick,sex,age_group,view \
    --intersect fitzpatrick,sex --min-n 15

## 6. Grad-CAM: ¿mira la lesión o el fondo?

En una foto de piel los atajos disponibles son distintos a los de una
radiografía: reglas de medición, marcas de bolígrafo, vello, sombras del borde
del encuadre.

In [ ]:
!python -m src.explain --checkpoint {OUT}/runs/pad_densenet121/best.pth \
    --manifest {OUT}/manifiesto_pad.csv --split test --n 200 \
    --out-dir {OUT}/reports/gradcam

# Se conservan unos pocos mapas como muestra y el JSON del audit
mapas = sorted(glob.glob(f'{OUT}/reports/gradcam/*.png'))
for p in mapas[12:]:
    os.remove(p)
print(f'{len(mapas)} mapas generados, {min(12, len(mapas))} conservados')

## 7. Resumen

In [ ]:
resumen = {'dataset': 'PAD-UFES-20', 'tarea': 'maligno (BCC/SCC/MEL) vs benigno (ACK/NEV/SEK)'}
for nombre, ruta in [('test', f'{OUT}/reports/test/metrics.json'),
                     ('biopsia', f'{OUT}/reports/biopsia.json'),
                     ('equidad', f'{OUT}/reports/equidad/fairness.json'),
                     ('gradcam', f'{OUT}/reports/gradcam/shortcut_audit.json')]:
    if os.path.exists(ruta):
        d = json.load(open(ruta))
        resumen[nombre] = {k: v for k, v in d.items() if k != 'detalle'}

brechas = resumen.get('equidad', {}).get('brechas', {})
if 'fitzpatrick' in brechas:
    b = brechas['fitzpatrick']
    resumen['titular'] = (f"Brecha de FNR entre tonos de piel: {b['brecha_fnr']:.4f} "
                          f"(peor tono: {b['peor_subgrupo']})")
    print('\n*** ' + resumen['titular'] + ' ***\n')

resumen['minutos'] = round((time.time() - T0) / 60, 1)
json.dump(resumen, open(f'{OUT}/resumen.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False)[:3000])

import shutil
for f_ in glob.glob(f'{OUT}/manifiesto_*.csv'):
    shutil.move(f_, '/tmp/' + os.path.basename(f_))